# 01 — Dataset overview

**Purpose.** Inventory of every dataset used in the study: size, features, positive rate, source, licence and integrity checksum.

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/01_dataset_overview_*.png|pdf` and `reports/notebooks/01_dataset_overview.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "01_dataset_overview"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Registry
Every dataset is declared once in `federated_learning.preprocessing.registry` with source, licence status, citation and SHA-256 of the files.

In [2]:
from federated_learning.preprocessing import SPECS
from federated_learning.preprocessing.real_datasets import load_real
from federated_learning.synthetic_data import make_dataset
rows = []
syn = make_dataset(n_samples=12000, seed=SEED)
rows.append(dict(dataset="SYN-IDS-v1 (synthetic)", samples=12000, features=40, positive_rate=round(float(np.r_[syn.y_train, syn.y_test].mean()), 3), license="n/a (generated)", setup="simulated IID / Dirichlet clients"))
for a in ("wdbc", "pima", "heart"):
    d = load_real(a); s = [x for x in SPECS.values() if x.alias == a][0]
    rows.append(dict(dataset=s.title, samples=d.meta["n_samples"], features=d.meta["n_features"], positive_rate=round(float(np.r_[d.y_train, d.y_test].mean()), 3), license=s.license, setup="simulated IID / Dirichlet clients"))
s4 = SPECS["uci_heart_disease_multisite"]
rows.append(dict(dataset=s4.title, samples=s4.n_samples, features=s4.n_features, positive_rate=None, license=s4.license, setup="4 natural site clients"))
tab = pd.DataFrame(rows); display(tab); SUMMARY["datasets"] = rows

,dataset,samples,features,positive_rate,license,setup
0,SYN-IDS-v1 (synthetic),12000,40,0.402,n/a (generated),simulated IID / Dirichlet clients
1,Breast Cancer Wisconsin (Diagnostic),569,30,0.627,CC BY 4.0,simulated IID / Dirichlet clients
2,Pima Indians Diabetes Database,768,8,0.349,not verified,simulated IID / Dirichlet clients
3,"Heart Disease (Cleveland, processed 14-attribu...",302,13,0.543,CC BY 4.0 (UCI Heart Disease dataset),simulated IID / Dirichlet clients
4,"Heart Disease, four collection sites (Clevelan...",920,10,NaN,CC BY 4.0 (UCI Heart Disease dataset),4 natural site clients


## 2. Integrity
SHA-256 values pinned in the registry (files are downloaded on first use and refused if the hash differs).

In [3]:
chk = pd.DataFrame([dict(dataset=k, file=f.filename, sha256=(f.sha256 or "")[:16] + "...") for k, s in SPECS.items() for f in s.files]); display(chk)

,dataset,file,sha256
0,wisconsin_breast_cancer,wdbc_sklearn_arrays.sha256,25e2eb7b7a8745b1...
1,pima_diabetes,pima.csv,6bfe5d0f379d17a0...
2,cleveland_heart_disease,heart.csv,e31e52eb5ee890c1...
3,uci_heart_disease_multisite,cleveland14.csv,a74b7efa387bc9d1...
4,uci_heart_disease_multisite,hungarian14r.csv,18b0496cd6171760...
5,uci_heart_disease_multisite,switzerland14.csv,834a405ccf5b66ab...
6,uci_heart_disease_multisite,long_beach_va14.csv,e7c93d8d0d2acdad...


## 3. Figure: size and positive rate

In [4]:
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.8))
t = tab.dropna(subset=["positive_rate"])
ax[0].barh(t.dataset.str.slice(0, 26), t.samples, color=PAL[0]); ax[0].set_xlabel("samples"); ax[0].set_title("Size")
ax[1].barh(t.dataset.str.slice(0, 26), t.positive_rate, color=PAL[1]); ax[1].set_xlabel("positive-class rate"); ax[1].set_title("Class balance"); ax[1].set_yticklabels([])
save_fig(fig, "overview"); save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/01_dataset_overview.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.